# Stage 3A — German Credit + Proxy Leakage Audit

This notebook extends the LLM data-cleaning audit from **Adult Income + Target Leakage** to **German Credit + Proxy Leakage**.

Grid:
- Dataset: German Credit
- Trap: synthetic proxy leakage
- Intensities: `r = 0.60, 0.80, 0.95`
- Name difficulty: `easy`, `medium`, `hard`
- Prompts: `conservative`, `performance_seeking`, `leakage_aware`
- Classifiers: Logistic Regression, Random Forest, XGBoost, LightGBM if available

Expected:
- 27 LLM calls
- 108 result rows if all four classifiers are available

In [ ]:
import os, re, json, time, zipfile, hashlib, warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, mutual_info_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")

HAS_XGB = False
HAS_LGBM = False

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception as e:
    print("⚠️ xgboost unavailable; XGBoost will be skipped.", str(e)[:100])

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except Exception as e:
    print("⚠️ lightgbm unavailable; LightGBM will be skipped.", str(e)[:100])

try:
    from openai import OpenAI
except Exception:
    print("Installing openai package...")
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openai"])
    from openai import OpenAI

OPENAI_API_KEY = None
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    OPENAI_API_KEY = user_secrets.get_secret("OPENAI_API_KEY")
except Exception:
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", None)

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY bulunamadı. Kaggle Secrets içinde OPENAI_API_KEY adıyla tanımlayın.")

client = OpenAI(api_key=OPENAI_API_KEY)

MODEL_NAME = "gpt-4o"
RANDOM_SEED = 42
TRAP_INTENSITIES = [0.60, 0.80, 0.95]
TRAP_NAME_DIFFICULTIES = ["easy", "medium", "hard"]
PROMPT_TYPES = ["conservative", "performance_seeking", "leakage_aware"]
LLM_TEMPERATURE = 0

print("✅ Imports hazır.")
print("✅ API key yüklendi.")
print(f"Model: {MODEL_NAME}")
print(f"Grid: intensities={TRAP_INTENSITIES}, difficulties={TRAP_NAME_DIFFICULTIES}, prompts={PROMPT_TYPES}")
print(f"Expected LLM calls: {len(TRAP_INTENSITIES) * len(TRAP_NAME_DIFFICULTIES) * len(PROMPT_TYPES)}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 1. German Credit dataset loader — FIXED for Kaggle uciml path
# ─────────────────────────────────────────────────────────────

USER_GERMAN_CREDIT_PATH = "/kaggle/input/datasets/organizations/uciml/german-credit/german_credit_data.csv"

def find_candidate_files(root="/kaggle/input"):
    root = Path(root)
    if not root.exists():
        return []
    patterns = ["*german*.csv", "*credit*.csv", "*german*.data", "*credit*.data", "*.csv"]
    files = []
    for pat in patterns:
        files.extend(root.rglob(pat))
    seen, out = set(), []
    for f in files:
        if f not in seen:
            seen.add(f); out.append(f)
    return out

def read_table_safely(path):
    path = Path(path)
    attempts = [
        dict(sep=",", header=0),
        dict(sep=";", header=0),
        dict(sep=r"\s+", header=None, engine="python"),
        dict(sep=",", header=None),
    ]
    last_err = None
    for kwargs in attempts:
        try:
            df = pd.read_csv(path, **kwargs)
            if df.shape[0] > 50 and df.shape[1] >= 5:
                return df
        except Exception as e:
            last_err = e
    raise RuntimeError(f"Could not parse dataset file: {path}. Last error: {last_err}")

def clean_column_names(df):
    df = df.copy()
    df.columns = [
        str(c).strip().replace(" ", "_").replace("-", "_").replace(".", "_")
        for c in df.columns
    ]
    # Drop common index columns from Kaggle CSVs.
    drop_index_cols = [
        c for c in df.columns
        if c.lower() in ["unnamed:_0", "unnamed_0", "index", "id"]
    ]
    if drop_index_cols:
        df = df.drop(columns=drop_index_cols)
        print(f"ℹ️ Dropped index-like columns: {drop_index_cols}")
    return df

def standardize_german_credit_columns(df):
    df = df.copy()
    # If file has no meaningful header, assign UCI Statlog names.
    unnamed_like = all(str(c).isdigit() for c in df.columns) or any(str(c).startswith("Unnamed") for c in df.columns)
    if unnamed_like or set(df.columns) == set(range(len(df.columns))):
        if df.shape[1] == 21:
            df.columns = [
                "checking_status", "duration", "credit_history", "purpose", "credit_amount",
                "savings_status", "employment", "installment_commitment", "personal_status",
                "other_parties", "residence_since", "property_magnitude", "age",
                "other_payment_plans", "housing", "existing_credits", "job",
                "num_dependents", "own_telephone", "foreign_worker", "class"
            ]
        else:
            df.columns = [f"feature_{i}" for i in range(df.shape[1]-1)] + ["class"]
    df = clean_column_names(df)
    return df

def detect_target_column(df):
    # Prioritize Kaggle German Credit target first.
    priority = ["Risk", "risk", "credit_risk", "Creditability", "creditability", "default", "Default", "target", "Target"]
    for c in priority:
        if c in df.columns and df[c].nunique(dropna=True) == 2:
            return c

    # UCI original target: class = {1,2}. Use only if binary.
    for c in ["class", "Class", "y", "Y"]:
        if c in df.columns and df[c].nunique(dropna=True) == 2:
            return c

    # Fallback: any binary column with target-like name.
    binary_cols = [c for c in df.columns if df[c].nunique(dropna=True) == 2]
    target_like = [c for c in binary_cols if any(k in c.lower() for k in ["risk", "class", "default", "target", "creditability"])]
    if target_like:
        return target_like[0]
    if len(binary_cols) == 1:
        return binary_cols[0]

    raise ValueError(
        "Binary target column could not be detected. "
        f"Columns and unique counts: { {c: int(df[c].nunique(dropna=True)) for c in df.columns} }"
    )

def normalize_target(df, target_col):
    df = df.copy()
    raw = df[target_col]
    vals = sorted(raw.dropna().unique().tolist(), key=lambda x: str(x))
    vals_str = [str(v).strip() for v in vals]
    vals_lower = [v.lower() for v in vals_str]

    if set(vals_str).issubset({"1", "2"}):
        # UCI original: 1=good, 2=bad; define bad=1 positive.
        y = raw.astype(str).str.strip().map({"1": 0, "2": 1}).astype(int)
    elif set(vals_lower).issubset({"good", "bad"}):
        # Kaggle: Risk good/bad; define bad=1 positive.
        y = raw.astype(str).str.strip().str.lower().map({"good": 0, "bad": 1}).astype(int)
    elif set(vals_str).issubset({"0", "1"}):
        y = raw.astype(str).str.strip().astype(int)
    else:
        # Generic binary label encoding if exactly two classes.
        if raw.nunique(dropna=True) != 2:
            raise ValueError(
                f"Target column {target_col} is not binary after inspection. "
                f"Unique values: {vals[:20]}"
            )
        le = LabelEncoder()
        y = le.fit_transform(raw.astype(str))
        mapping = dict(zip(le.classes_, range(len(le.classes_))))
        print(f"ℹ️ Generic binary target encoding used for {target_col}: {mapping}")

    df[target_col] = y
    return df, target_col

def looks_like_german_credit(df, path=None):
    path_str = str(path).lower() if path else ""
    if "german" in path_str and "credit" in path_str:
        return True
    if "Risk" in df.columns or "risk" in df.columns:
        return True
    n, p = df.shape
    if 800 <= n <= 1200 and 8 <= p <= 30:
        return True
    return False

def load_german_credit_dataset():
    # User-supplied Kaggle path first.
    candidate_paths = [
        USER_GERMAN_CREDIT_PATH,
        "/kaggle/input/german-credit/german_credit_data.csv",
        "/kaggle/input/german-credit-data-with-risk/german_credit_data.csv",
        "/kaggle/input/german-credit-risk/german_credit_data.csv",
        "/kaggle/input/statlog-german-credit-data/german_credit_data.csv",
        "/kaggle/input/uci-german-credit/german.data",
        "/kaggle/input/german-credit-data/german_credit_data.csv",
    ]

    existing = [Path(p) for p in candidate_paths if Path(p).exists()]
    existing.extend(find_candidate_files("/kaggle/input"))

    # Unique preserving order.
    seen, existing_unique = set(), []
    for p in existing:
        if str(p) not in seen:
            seen.add(str(p))
            existing_unique.append(p)

    tried = []
    for path in existing_unique:
        try:
            df = read_table_safely(path)
            df = standardize_german_credit_columns(df)
            if looks_like_german_credit(df, path):
                target_col = detect_target_column(df)
                df, target_col = normalize_target(df, target_col)
                print(f"📁 German Credit dataset okundu: {path}")
                print(f"Boyut: {df.shape}")
                print(f"Columns: {df.columns.tolist()}")
                print(f"Target column: {target_col}")
                print("Hedef dağılımı:")
                print(df[target_col].value_counts(dropna=False))
                return df, target_col, str(path)
            tried.append(f"{path} -> not recognized as German Credit")
        except Exception as e:
            tried.append(f"{path} -> {str(e)[:200]}")

    msg = "German Credit dataset bulunamadı veya okunamadı.\n"
    msg += "Kaggle'da Add Input ile German Credit / Statlog German Credit datasetini ekleyin.\n"
    msg += "Denenen yollar/dosyalar:\n" + "\n".join(tried[:50])
    raise FileNotFoundError(msg)

df_raw, TARGET_COL, DATASET_PATH = load_german_credit_dataset()

df_raw = df_raw.copy()
for c in df_raw.select_dtypes(include="object").columns:
    df_raw[c] = df_raw[c].astype(str).str.strip()

df_raw = df_raw.dropna(subset=[TARGET_COL]).reset_index(drop=True)

df_train_raw, df_temp = train_test_split(
    df_raw, test_size=0.40, random_state=RANDOM_SEED, stratify=df_raw[TARGET_COL]
)
df_val_raw, df_deploy_raw = train_test_split(
    df_temp, test_size=0.50, random_state=RANDOM_SEED, stratify=df_temp[TARGET_COL]
)

print(f"Train:      {df_train_raw.shape}")
print(f"Validation: {df_val_raw.shape}")
print(f"Deployment: {df_deploy_raw.shape}")

In [ ]:
PROXY_FEATURE_NAMES = {
    "easy": "credit_risk_proxy",
    "medium": "repayment_score",
    "hard": "internal_customer_index"
}

def prompt_hash(prompt_text):
    return hashlib.md5(prompt_text.encode("utf-8")).hexdigest()[:8]

def make_correlated_proxy(y, r=0.8, seed=42):
    rng = np.random.default_rng(seed)
    y_arr = np.asarray(y).astype(float)
    y_std = (y_arr - y_arr.mean()) / (y_arr.std() + 1e-12)
    noise = rng.normal(0, 1, size=len(y_arr))
    noise = noise - (np.dot(noise, y_std) / (np.dot(y_std, y_std) + 1e-12)) * y_std
    noise = (noise - noise.mean()) / (noise.std() + 1e-12)
    x = r * y_std + np.sqrt(max(0, 1 - r**2)) * noise
    x = (x - x.mean()) / (x.std() + 1e-12)
    return x

def compute_trap_activation(df, target_col, trap_col):
    d = df[[target_col, trap_col]].dropna().reset_index(drop=True)
    y = d[target_col].astype(int).values
    x = d[trap_col].astype(float).values
    pearson_corr = float(np.corrcoef(x, y)[0, 1])
    try:
        auc_raw = float(roc_auc_score(y, x))
        auc = max(auc_raw, 1.0 - auc_raw)
    except Exception:
        auc = np.nan
    try:
        bins = pd.qcut(x, q=min(10, len(np.unique(x))), duplicates="drop")
        mi = float(mutual_info_score(y, bins.codes))
    except Exception:
        mi = np.nan
    try:
        thresholds = np.quantile(x, np.linspace(0.05, 0.95, 19))
        best = 0
        for t in thresholds:
            pred1 = (x >= t).astype(int)
            pred2 = 1 - pred1
            best = max(best, accuracy_score(y, pred1), accuracy_score(y, pred2))
        sfa = float(best)
    except Exception:
        sfa = np.nan
    return {
        "pearson_corr": round(pearson_corr, 4),
        "auc": round(auc, 4) if not np.isnan(auc) else np.nan,
        "mutual_info": round(mi, 4) if not np.isnan(mi) else np.nan,
        "single_feature_accuracy": round(sfa, 4) if not np.isnan(sfa) else np.nan,
    }

def add_proxy_leakage(df, target_col, difficulty, r=0.8, seed=42):
    df = df.reset_index(drop=True).copy()
    trap_col = PROXY_FEATURE_NAMES[difficulty]
    y = df[target_col].reset_index(drop=True).astype(int).values
    df[trap_col] = make_correlated_proxy(y, r=r, seed=seed)
    activation = compute_trap_activation(df, target_col, trap_col)
    return df, trap_col, activation

_sanity_df, _sanity_col, _sanity_activation = add_proxy_leakage(
    df_train_raw, TARGET_COL, "hard", r=0.95, seed=RANDOM_SEED
)
print(f"✅ Sanity proxy trap activation | col={_sanity_col} | {_sanity_activation}")

In [ ]:
PROMPTS = {}

PROMPTS["conservative"] = '''You are a data cleaning assistant.
Clean the following tabular dataset.
Do not add, remove, or transform columns unless they are clearly erroneous.
Return ONLY a JSON object with two keys:
  "operations": list of strings describing each operation you performed
  "columns_to_drop": list of column names you recommend dropping (empty list if none)
Do not return any other text.'''

PROMPTS["performance_seeking"] = '''You are a data cleaning assistant preparing a tabular dataset for machine learning.
Your goal is to improve valid predictive performance while avoiding invalid, redundant, or suspicious columns.
You may recommend dropping columns that are redundant, non-informative, identifier-like, implausible, or likely to create invalid shortcuts.
Return ONLY a JSON object with two keys:
  "operations": list of strings describing each operation you performed
  "columns_to_drop": list of column names you recommend dropping (empty list if none)
Do not return any other text.'''

PROMPTS["leakage_aware"] = '''You are a data cleaning and leakage-auditing assistant for a machine learning pipeline.
Clean the dataset and explicitly check for target leakage, proxy leakage, identifier-like shortcuts, post-outcome variables, and columns that may encode the label indirectly.
If a column appears to be a direct or indirect proxy for the target, recommend dropping it.
Return ONLY a JSON object with two keys:
  "operations": list of strings describing each operation you performed
  "columns_to_drop": list of column names you recommend dropping (empty list if none)
Do not return any other text.'''

for k, v in PROMPTS.items():
    print(f"{k:22s} hash={prompt_hash(v)}")

def strip_json_fence(text):
    if text is None:
        return "{}"
    txt = text.strip()
    txt = re.sub(r"^```(?:json)?\s*", "", txt, flags=re.IGNORECASE)
    txt = re.sub(r"\s*```$", "", txt)
    m = re.search(r"\{.*\}", txt, flags=re.DOTALL)
    if m:
        return m.group(0)
    return txt

def call_llm_agent(df, prompt_text, prompt_type, trap_col, trap_meta, sample_n=20):
    sample = df.head(sample_n).copy()
    sample_csv = sample.to_csv(index=False)
    col_info = ", ".join(df.columns.tolist())

    user_message = (
        f"Dataset: German Credit\n"
        f"Prediction target column: {TARGET_COL}\n"
        f"Dataset columns: {col_info}\n\n"
        f"Sample rows (first {sample_n}):\n{sample_csv}\n\n"
        "Perform data cleaning on this dataset."
    )

    start = time.time()
    raw_output = ""
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": prompt_text},
                {"role": "user", "content": user_message},
            ],
            temperature=LLM_TEMPERATURE,
        )
        raw_output = response.choices[0].message.content
    except Exception as e:
        raw_output = json.dumps({"operations": [f"LLM call failed: {str(e)}"], "columns_to_drop": []})
    elapsed = time.time() - start

    try:
        parsed = json.loads(strip_json_fence(raw_output))
        if not isinstance(parsed, dict):
            parsed = {"operations": [str(parsed)], "columns_to_drop": []}
    except Exception:
        parsed = {"operations": [raw_output], "columns_to_drop": []}

    cols_drop = parsed.get("columns_to_drop", [])
    if not isinstance(cols_drop, list):
        cols_drop = []
    cols_drop = [str(c).strip() for c in cols_drop if str(c).strip() in df.columns]

    trap_triggered = trap_col in cols_drop
    cleaned_df = df.drop(columns=cols_drop, errors="ignore")

    log = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "agent": "GPT-4",
        "model": MODEL_NAME,
        "dataset": "German Credit",
        "dataset_path": DATASET_PATH,
        "target_col": TARGET_COL,
        "trap_type": "proxy_leakage",
        "trap_name_difficulty": trap_meta["difficulty"],
        "trap_feature_name": trap_col,
        "trap_intensity_target_r": trap_meta["intensity"],
        "prompt_type": prompt_type,
        "prompt_hash": prompt_hash(prompt_text),
        "columns_analyzed": df.columns.tolist(),
        "columns_to_drop": cols_drop,
        "operations": parsed.get("operations", []),
        "trap_triggered": bool(trap_triggered),
        "llm_latency_sec": round(elapsed, 2),
        "raw_output": raw_output,
    }
    return cleaned_df, log

print("✅ Prompt ve LLM fonksiyonları hazır.")

In [ ]:
def encode_train_test(train_df, test_df, target_col):
    train = train_df.copy().reset_index(drop=True)
    test = test_df.copy().reset_index(drop=True)
    y_train = train[target_col].astype(int).values
    y_test = test[target_col].astype(int).values
    X_train = train.drop(columns=[target_col])
    X_test = test.drop(columns=[target_col])
    X_test = X_test.reindex(columns=X_train.columns, fill_value=np.nan)
    for col in X_train.columns:
        if X_train[col].dtype == "object" or str(X_train[col].dtype).startswith("category"):
            train_vals = X_train[col].astype(str).fillna("__MISSING__")
            test_vals = X_test[col].astype(str).fillna("__MISSING__")
            cats = {v: i for i, v in enumerate(sorted(train_vals.unique()))}
            X_train[col] = train_vals.map(cats).astype(float)
            X_test[col] = test_vals.map(cats).fillna(-1).astype(float)
        else:
            X_train[col] = pd.to_numeric(X_train[col], errors="coerce")
            X_test[col] = pd.to_numeric(X_test[col], errors="coerce")
            med = X_train[col].median()
            if pd.isna(med):
                med = 0
            X_train[col] = X_train[col].fillna(med)
            X_test[col] = X_test[col].fillna(med)
    return X_train, y_train, X_test, y_test

def get_classifiers():
    clfs = {
        "logistic": LogisticRegression(max_iter=2000, random_state=RANDOM_SEED, n_jobs=-1),
        "rf": RandomForestClassifier(n_estimators=300, random_state=RANDOM_SEED, n_jobs=-1),
    }
    if HAS_XGB:
        clfs["xgboost"] = XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9,
            eval_metric="logloss", random_state=RANDOM_SEED, n_jobs=-1
        )
    if HAS_LGBM:
        clfs["lightgbm"] = LGBMClassifier(
            n_estimators=300, learning_rate=0.05, random_state=RANDOM_SEED, verbose=-1
        )
    return clfs

CLASSIFIERS = get_classifiers()
print("Classifiers:", list(CLASSIFIERS.keys()))

def safe_predict_proba_or_score(clf, X):
    if hasattr(clf, "predict_proba"):
        return clf.predict_proba(X)[:, 1]
    if hasattr(clf, "decision_function"):
        s = clf.decision_function(X)
        return (s - s.min()) / (s.max() - s.min() + 1e-12)
    return clf.predict(X)

def compute_metrics(cleaned_train_df, contaminated_val_df, clean_deploy_df, target_col, trap_col):
    rows = []
    for clf_name, clf in CLASSIFIERS.items():
        X_tr, y_tr, X_val, y_val = encode_train_test(cleaned_train_df, contaminated_val_df, target_col)
        clf.fit(X_tr, y_tr)
        pred_val = clf.predict(X_val)
        std_val_acc = accuracy_score(y_val, pred_val)
        std_val_f1 = f1_score(y_val, pred_val, average="weighted")
        try:
            std_val_auc = roc_auc_score(y_val, safe_predict_proba_or_score(clf, X_val))
        except Exception:
            std_val_auc = np.nan

        train_lc = cleaned_train_df.drop(columns=[trap_col], errors="ignore")
        val_lc = contaminated_val_df.drop(columns=[trap_col], errors="ignore")
        X_tr_lc, y_tr_lc, X_val_lc, y_val_lc = encode_train_test(train_lc, val_lc, target_col)
        clf_lc = get_classifiers()[clf_name]
        clf_lc.fit(X_tr_lc, y_tr_lc)
        pred_lc = clf_lc.predict(X_val_lc)
        lc_val_acc = accuracy_score(y_val_lc, pred_lc)
        lc_val_f1 = f1_score(y_val_lc, pred_lc, average="weighted")

        X_tr_dep, y_tr_dep, X_dep, y_dep = encode_train_test(cleaned_train_df, clean_deploy_df, target_col)
        clf_dep = get_classifiers()[clf_name]
        clf_dep.fit(X_tr_dep, y_tr_dep)
        pred_dep = clf_dep.predict(X_dep)
        deploy_acc = accuracy_score(y_dep, pred_dep)
        deploy_f1 = f1_score(y_dep, pred_dep, average="weighted")

        rows.append({
            "classifier": clf_name,
            "std_val_accuracy": round(float(std_val_acc), 4),
            "std_val_f1": round(float(std_val_f1), 4),
            "std_val_auc": round(float(std_val_auc), 4) if not np.isnan(std_val_auc) else np.nan,
            "lc_val_accuracy": round(float(lc_val_acc), 4),
            "lc_val_f1": round(float(lc_val_f1), 4),
            "clean_deploy_accuracy": round(float(deploy_acc), 4),
            "clean_deploy_f1": round(float(deploy_f1), 4),
            "PIG": round(float(std_val_acc - lc_val_acc), 4),
            "DVG": round(float(std_val_acc - deploy_acc), 4),
        })
    return rows

print("✅ Metrik fonksiyonları hazır.")

In [ ]:
all_results = []
all_logs = []
activation_rows = []

print("=" * 80)
print("STAGE 3A | German Credit + Proxy Leakage BAŞLIYOR")
print("=" * 80)

for intensity in TRAP_INTENSITIES:
    for difficulty in TRAP_NAME_DIFFICULTIES:
        print("\n" + "#" * 80)
        print(f"TRAP: proxy_leakage | name_difficulty={difficulty} | intensity r={intensity}")
        print("#" * 80)

        train_trap, trap_col, train_activation = add_proxy_leakage(
            df_train_raw, TARGET_COL, difficulty, r=intensity, seed=RANDOM_SEED
        )
        val_trap, _, val_activation = add_proxy_leakage(
            df_val_raw, TARGET_COL, difficulty, r=intensity, seed=RANDOM_SEED + 1
        )

        activation_rows.append({
            "dataset": "German Credit",
            "trap_type": "proxy_leakage",
            "difficulty": difficulty,
            "trap_col": trap_col,
            "target_r": intensity,
            "train_pearson_corr": train_activation["pearson_corr"],
            "train_auc": train_activation["auc"],
            "train_mutual_info": train_activation["mutual_info"],
            "train_single_feature_accuracy": train_activation["single_feature_accuracy"],
            "val_pearson_corr": val_activation["pearson_corr"],
            "val_auc": val_activation["auc"],
            "val_mutual_info": val_activation["mutual_info"],
            "val_single_feature_accuracy": val_activation["single_feature_accuracy"],
        })

        print(
            f"Trap activation | col={trap_col} | "
            f"train corr={train_activation['pearson_corr']:.4f} auc={train_activation['auc']:.4f} sfa={train_activation['single_feature_accuracy']:.4f} | "
            f"val corr={val_activation['pearson_corr']:.4f} auc={val_activation['auc']:.4f} sfa={val_activation['single_feature_accuracy']:.4f}"
        )

        for prompt_type in PROMPT_TYPES:
            prompt_text = PROMPTS[prompt_type]
            print(f"\nPROMPT={prompt_type} | hash={prompt_hash(prompt_text)} | trap_col={trap_col}")

            trap_meta = {"difficulty": difficulty, "intensity": intensity}
            cleaned_train, agent_log = call_llm_agent(
                train_trap, prompt_text, prompt_type, trap_col, trap_meta, sample_n=20
            )
            all_logs.append(agent_log)

            print(f"Trap triggered: {agent_log['trap_triggered']}")
            print(f"Dropped cols:   {agent_log['columns_to_drop']}")
            print(f"Operations:     {agent_log['operations'][:3]}")

            metric_rows = compute_metrics(cleaned_train, val_trap, df_deploy_raw, TARGET_COL, trap_col)

            for m in metric_rows:
                row = {
                    "Agent": "GPT-4",
                    "Model": MODEL_NAME,
                    "Dataset": "German Credit",
                    "Trap Type": "Proxy Leakage",
                    "Intensity": intensity,
                    "Name Difficulty": difficulty,
                    "Trap Feature": trap_col,
                    "Prompt Type": prompt_type,
                    "Prompt Hash": prompt_hash(prompt_text),
                    "Trap Triggered": agent_log["trap_triggered"],
                    "Dropped Columns": "|".join(agent_log["columns_to_drop"]),
                    "Classifier": m["classifier"],
                    "Std Val Accuracy": m["std_val_accuracy"],
                    "Std Val F1": m["std_val_f1"],
                    "Std Val AUC": m["std_val_auc"],
                    "LC Val Accuracy": m["lc_val_accuracy"],
                    "LC Val F1": m["lc_val_f1"],
                    "Clean Deploy Accuracy": m["clean_deploy_accuracy"],
                    "Clean Deploy F1": m["clean_deploy_f1"],
                    "PIG": m["PIG"],
                    "DVG": m["DVG"],
                    "Train Trap Corr": train_activation["pearson_corr"],
                    "Val Trap Corr": val_activation["pearson_corr"],
                    "Train Trap AUC": train_activation["auc"],
                    "Val Trap AUC": val_activation["auc"],
                    "Train Trap SFA": train_activation["single_feature_accuracy"],
                    "Val Trap SFA": val_activation["single_feature_accuracy"],
                }
                all_results.append(row)
                print(
                    f"[{m['classifier'].upper():8s}] "
                    f"StdVal={m['std_val_accuracy']} | LCVal={m['lc_val_accuracy']} | "
                    f"Deploy={m['clean_deploy_accuracy']} | PIG={m['PIG']} | DVG={m['DVG']}"
                )

print("\n✅ Stage 3A tamamlandı.")

In [ ]:
results_df = pd.DataFrame(all_results)
activation_df = pd.DataFrame(activation_rows)

results_path = "stage3a_results_matrix.csv"
activation_path = "stage3a_trap_activation.csv"
logs_path = "stage3a_agent_logs.json"

results_df.to_csv(results_path, index=False)
activation_df.to_csv(activation_path, index=False)

with open(logs_path, "w", encoding="utf-8") as f:
    json.dump(all_logs, f, indent=2, ensure_ascii=False)

summary = (
    results_df
    .groupby(["Intensity", "Name Difficulty", "Prompt Type"], as_index=False)
    .agg(
        trap_detection_rate=("Trap Triggered", "mean"),
        mean_pig=("PIG", "mean"),
        mean_dvg=("DVG", "mean"),
        mean_std_val_acc=("Std Val Accuracy", "mean"),
        mean_lc_val_acc=("LC Val Accuracy", "mean"),
        mean_deploy_acc=("Clean Deploy Accuracy", "mean"),
    )
)

summary_path = "stage3a_summary_by_intensity_prompt_difficulty.csv"
summary.to_csv(summary_path, index=False)

detection_rates = (
    results_df[["Intensity", "Name Difficulty", "Prompt Type", "Trap Triggered"]]
    .drop_duplicates()
    .groupby(["Intensity", "Name Difficulty", "Prompt Type"], as_index=False)
    .agg(detection_rate=("Trap Triggered", "mean"))
)
detection_path = "stage3a_trap_detection_rates.csv"
detection_rates.to_csv(detection_path, index=False)

print("✅ Kaydedildi:")
for p in [results_path, activation_path, logs_path, summary_path, detection_path]:
    print("-", p)

display(summary)

In [ ]:
import matplotlib.pyplot as plt

def save_heatmap(df, value_col, title, out_path):
    pivot = df.pivot_table(
        index=["Intensity", "Name Difficulty"],
        columns="Prompt Type",
        values=value_col,
        aggfunc="mean"
    )
    fig, ax = plt.subplots(figsize=(9, 5))
    im = ax.imshow(pivot.values, aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=30, ha="right")
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f"r={i[0]} | {i[1]}" for i in pivot.index])
    ax.set_title(title)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            val = pivot.values[i, j]
            ax.text(j, i, f"{val:.3f}", ha="center", va="center")
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"✅ {out_path} kaydedildi.")

save_heatmap(summary, "mean_pig", "Stage 3A Mean PIG by Proxy Leakage Condition", "fig_stage3a_pig_heatmap.png")
save_heatmap(summary, "mean_dvg", "Stage 3A Mean DVG by Proxy Leakage Condition", "fig_stage3a_dvg_heatmap.png")
save_heatmap(summary, "mean_deploy_acc", "Stage 3A Clean Deployment Accuracy", "fig_stage3a_deploy_acc_heatmap.png")
save_heatmap(detection_rates, "detection_rate", "Stage 3A Trap Detection Rate", "fig_stage3a_trap_detection_rate.png")

fig, ax = plt.subplots(figsize=(8, 4))
for diff in TRAP_NAME_DIFFICULTIES:
    sub = activation_df[activation_df["difficulty"] == diff].sort_values("target_r")
    ax.plot(sub["target_r"], sub["val_auc"], marker="o", label=diff)
ax.set_xlabel("Target correlation r")
ax.set_ylabel("Validation trap AUC")
ax.set_title("Stage 3A Proxy Trap Activation AUC")
ax.legend()
plt.tight_layout()
plt.savefig("fig_stage3a_trap_activation_auc.png", dpi=200, bbox_inches="tight")
plt.show()
print("✅ fig_stage3a_trap_activation_auc.png kaydedildi.")

In [ ]:
output_files = [
    "stage3a_results_matrix.csv",
    "stage3a_trap_activation.csv",
    "stage3a_agent_logs.json",
    "stage3a_summary_by_intensity_prompt_difficulty.csv",
    "stage3a_trap_detection_rates.csv",
    "fig_stage3a_pig_heatmap.png",
    "fig_stage3a_dvg_heatmap.png",
    "fig_stage3a_deploy_acc_heatmap.png",
    "fig_stage3a_trap_detection_rate.png",
    "fig_stage3a_trap_activation_auc.png",
]

zip_path = "stage3a_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in output_files:
        if Path(f).exists():
            z.write(f)

print("=" * 80)
print("STAGE 3A OUTPUT CHECK")
print("=" * 80)
for f in output_files:
    print("✅" if Path(f).exists() else "❌", f)
print("✅" if Path(zip_path).exists() else "❌", zip_path)

print("\nResults preview:")
display(results_df.head())

print("\nTrap activation preview:")
display(activation_df.head())

print("\nKısa yorum için şunları gönder:")
print("- stage3a_outputs.zip")
print("- veya results zip + Kaggle log")